# 4-Way Architecture Screening: Full Local Evaluation (No Time Cap)

**Companion to `DLDOA_Architecture_Screening_4Way.ipynb` (the Kaggle, time-boxed version).**
That notebook exists to answer "is this worth investing in, cheaply" inside a hard 2.5h
Kaggle session. This notebook drops the wall-clock cap entirely and trains each architecture
for a fixed, full epoch count on your own machine, then evaluates on the **complete** frozen
eval bank (8000 samples, not a subsample) -- meant to run for many hours unattended.

## Read this before running

**GPU on native Windows will not work.** TensorFlow >= 2.11 does not support GPU on native
Windows at all (Microsoft/Google dropped it), regardless of what NVIDIA GPU or drivers are
installed -- confirmed directly while building this notebook (`pip install tensorflow`
prints this warning verbatim on Windows). If this machine is Linux or Mac with a working
CUDA (Linux) or Metal (Apple Silicon, via `tensorflow-metal`) setup, GPU will be picked up
automatically by Cell 1 below with no extra config. If it's native Windows, this will fall
back to CPU silently -- check Cell 1's printed GPU list before trusting a training-time
estimate.

**A real, project-blocking bug was caught and fixed here.** The original 4 architecture
bodies (as written for the Kaggle notebook) used raw `tf.sin` / `tf.stack` / `tf.reshape`
calls directly on a Keras Functional-API tensor, and one of them (`GridlessUnfold`) created
a raw `tf.Variable` inside a plain Python function instead of through a Layer. Keras 3
(bundled with TensorFlow >= 2.16, which is what a fresh `pip install tensorflow` gives you
today, on this machine or Kaggle's current image) rejects the first pattern outright with a
`KerasTensor cannot be used as input to a TensorFlow function` error, and would have silently
under-trained or not-trained the `GridlessUnfold` threshold parameter for the second reason
even where it didn't error. All 3 affected blocks (SIREN, Window-Attention, Gridless-Unfold)
were rewritten as proper `Layer` subclasses / `Lambda`-wrapped ops -- verified end-to-end
locally on CPU (build -> forward -> backward/gradient step -> checkpoint save, all 4
architectures) before this notebook was written. **The same fix should be back-ported to
`DLDOA_Architecture_Screening_4Way.ipynb` before that one is ever run on Kaggle**, or it will
hit the same error.

**Training time is genuinely unknown until you run it.** No real per-step timing exists yet
for these 4 architectures on any hardware (the Kaggle version has never been run either).
Rather than guess, `train_architecture()` below measures real per-epoch time on **your**
machine after epoch 1 and prints a projected total for the configured epoch count --
watch that line before deciding whether to let the whole 4-architecture run go unattended
overnight or across several days.

**Crash/interruption resilience.** A run at this scale (hours to days, unattended) will
likely get interrupted at some point (laptop sleep, restart, power loss). Each architecture
checkpoints its weights periodically during training and writes a `<name>.done.json` marker
when it finishes; re-running the notebook from the top skips any architecture whose marker
already exists (loads its saved weights and goes straight to evaluating it) instead of
re-training from scratch.

## Files you need on this machine, and where

No large dataset files are needed -- training data is generated synthetically on the fly by
`training_data_generator`, not read from disk. You only need:

| File | Approx. size | Purpose |
|---|---|---|
| `DL_DOA/src/tvt_models.py` | 8 KB | `Resnet` class (teacher reference only) |
| `DL_DOA/src/tvt_data_generation_v3.py` | 20 KB | transitive import of the evaluator below |
| `DL_DOA/src/TVT_Blob_Inference.py` | 15 KB | blob-detector evaluator (Pd/RMSE) |
| `dldoa_dataset_generation.py` (repo root, **not** inside `DL_DOA/`) | 44 KB | `training_data_generator` |
| `frozen_banks/eval_bank.npz` | ~19 MB | **required** -- the fixed 8000-sample eval set |
| `DL_DOA/models/inf_model_007_256_resnet.h5` | ~3 MB | optional -- teacher reference row only |

Total ≈ 22 MB. Easiest approach: copy this whole repo's layout (or just these paths, keeping
them in the same relative positions to each other) onto the other machine, and put this
notebook in a `notebooks/` folder alongside `DL_DOA/`, `frozen_banks/`, and
`dldoa_dataset_generation.py`, matching the structure here. `find_path()` in Cell 2 searches
a few candidate roots so exact placement isn't critical as long as the relative structure
holds.

## Packages needed on the other machine
```
pip install tensorflow opencv-python-headless scipy matplotlib tqdm h5py
```
On Linux with an NVIDIA GPU, the plain `tensorflow` wheel includes CUDA support. On Apple
Silicon Mac, use `pip install tensorflow-macos tensorflow-metal` instead for GPU support.

## Config you'll likely want to change (Cell 5)
- `EPOCHS_PER_ARCH` (default 200) x `STEPS_PER_EPOCH` (default 100) = 20,000 gradient steps
  per architecture, from-scratch (8 blocks, no pretrained warm-start -- none of these 4
  candidates has one). This is a deliberately large, "let it run" scale, not a paper-scale
  claim -- the same honest-scope caveats as the Kaggle notebook still apply: equal *epoch
  count* here (not equal wall-clock like the Kaggle version), but FNO/Window-Attention still
  cost more per step than SIREN/Gridless-Unfold, so they'll simply take longer, not train
  less -- a fairer comparison than the Kaggle version's fixed-wall-clock design, at the cost
  of an unknown total run time until Cell 1's GPU check + the first measured ETA are in.


In [ ]:
# Cell 1 -- Setup, GPU detection, wall-clock (for logging/ETA only, NOT a time cap)
import importlib, subprocess, sys, time
T_START = time.time()

try:
    import cv2
except ImportError:
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'opencv-python-headless'], check=True)
    import cv2

import os, json
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow.keras.layers import Conv2D, Input, BatchNormalization, Activation, Add, Conv2DTranspose, Layer, Lambda
from tensorflow.keras.models import Model

tf.keras.backend.clear_session()
tf.get_logger().setLevel('ERROR')
np.random.seed(42); tf.random.set_seed(42)

gpus = tf.config.list_physical_devices('GPU')
print(f'TF: {tf.__version__}  |  GPU: {gpus}')
if not gpus:
    print('No GPU visible to TensorFlow. On native Windows this is EXPECTED (TF >=2.11 has no')
    print('native-Windows GPU support at all) -- training will run on CPU and take much longer')
    print('than the measured-ETA projections below would suggest on a real GPU machine.')
for g in gpus:
    tf.config.experimental.set_memory_growth(g, True)

OUT_DIR = os.path.join(os.getcwd(), 'outputs_full_eval')
CKPT_DIR = os.path.join(OUT_DIR, 'checkpoints')
os.makedirs(CKPT_DIR, exist_ok=True)

def elapsed_hours():
    return (time.time() - T_START) / 3600

print('Wall clock started. t=0.00h  |  outputs ->', OUT_DIR)


In [ ]:
# Cell 2 -- Locate source files, teacher weights, frozen eval bank, training generator.
# No Kaggle-specific paths -- searches a few generic local roots instead.
def find_path(name_pattern, roots):
    from pathlib import Path
    for root in roots:
        if not os.path.isdir(root):
            continue
        for p in Path(root).rglob(name_pattern):
            if p.is_file():
                return str(p)
    return None

NOTEBOOK_DIR = os.getcwd()
SEARCH_ROOTS = [NOTEBOOK_DIR, os.path.dirname(NOTEBOOK_DIR), os.path.dirname(os.path.dirname(NOTEBOOK_DIR))]

SRC_MODEL_FILE = find_path('tvt_models.py', SEARCH_ROOTS)
assert SRC_MODEL_FILE is not None, (
    'DL_DOA source not found. Copy the DL_DOA/ folder (see Cell 0 for the exact file list) '
    'somewhere under, next to, or one level above this notebook.')
DL_DOA_DIR = os.path.dirname(os.path.dirname(SRC_MODEL_FILE))
print(f'DL_DOA dir: {DL_DOA_DIR}')

TEACHER_WEIGHTS_PATH = find_path('inf_model_007_256_resnet.h5', SEARCH_ROOTS)
EVAL_BANK_PATH = find_path('eval_bank.npz', SEARCH_ROOTS)
assert EVAL_BANK_PATH is not None, 'eval_bank.npz not found -- copy frozen_banks/eval_bank.npz over'
TRAIN_GEN_FILE = find_path('dldoa_dataset_generation.py', SEARCH_ROOTS)
assert TRAIN_GEN_FILE is not None, 'dldoa_dataset_generation.py not found'
print(f'Teacher weights: {TEACHER_WEIGHTS_PATH}')
print(f'Eval bank: {EVAL_BANK_PATH}')
print(f'Training generator: {TRAIN_GEN_FILE}')


In [ ]:
# Cell 3 -- Import the ORIGINAL evaluator + Resnet + training generator (unmodified,
# same source used by every other notebook in this project)
sys.path.insert(0, DL_DOA_DIR)
sys.path.insert(0, os.path.dirname(TRAIN_GEN_FILE))
from src.tvt_models import Resnet
from src.TVT_Blob_Inference import get_blob_detector, get_blob_peaks, peaks_to_angles, prepare_for_metric, get_ang_difference, filter_angles
from dldoa_dataset_generation import training_data_generator
print('Imports OK')


In [ ]:
# Cell 4 -- Load the FULL frozen eval bank (8000 samples, no subsampling -- this is the
# "fully evaluate" run) + teacher (reference row only, never modified)
eval_bank = np.load(EVAL_BANK_PATH)
EVAL_DATA, EVAL_FEAT, EVAL_META = eval_bank['data'], eval_bank['feat'], eval_bank['meta']
SIGMA = float(eval_bank['sigma']); M = int(eval_bank['M'])
print(f'Eval bank: {EVAL_DATA.shape[0]} samples, SIGMA={SIGMA}, M={M}')

teacher = None
if TEACHER_WEIGHTS_PATH:
    teacher = Resnet(input_shape=(64, 64, 2))
    teacher.load_weights(TEACHER_WEIGHTS_PATH)
    teacher.trainable = False
    print(f'Teacher loaded: {teacher.count_params():,} params (reference only)')
else:
    print('Teacher weights not found -- proceeding without a reference row')


In [ ]:
# Cell 5 -- CONFIG
N_BLOCKS = 8
FILTERS = 12
EPOCHS_PER_ARCH = 200      # no wall-clock cap -- fixed epoch count per architecture instead
STEPS_PER_EPOCH = 100      # 200 x 100 = 20,000 gradient steps/architecture, from scratch
CHECKPOINT_EVERY_EPOCHS = 10   # periodic weight save during training (crash resilience)
ETA_AFTER_EPOCH = 1        # print a measured (not guessed) time projection after this many epochs

N_PER_SNR_EVAL = None      # None = full bank (1000/SNR x 8 SNRs = 8000 samples). Set an int
                           # (e.g. 100) to evaluate on a subsample instead, for a quicker look.

print('N_BLOCKS=', N_BLOCKS, ' FILTERS=', FILTERS)
print('EPOCHS_PER_ARCH=', EPOCHS_PER_ARCH, ' STEPS_PER_EPOCH=', STEPS_PER_EPOCH,
      ' -> ', EPOCHS_PER_ARCH * STEPS_PER_EPOCH, 'gradient steps/architecture')
print('Checkpoint every', CHECKPOINT_EVERY_EPOCHS, 'epochs')
print('Eval set size:', 'FULL (8000)' if N_PER_SNR_EVAL is None else f'{N_PER_SNR_EVAL}/SNR subsample')


In [ ]:
# Cell 6 -- Shared I/O wrapper + FNO's custom spectral-conv layer
# All 4 architectures plug a "body_fn(x, filters, n_blocks)" into this exact same shell,
# so only the core block differs -- everything else (input/output size, training loop,
# evaluator) is identical across all 4.

def build_model_with_body(body_fn, filters=FILTERS, n_blocks=N_BLOCKS, name='model'):
    x_in = Input(shape=(64, 64, 2))
    x = Conv2DTranspose(filters, (5, 5), strides=(2, 2), padding='same')(x_in)
    x = body_fn(x, filters, n_blocks)
    x = Conv2DTranspose(1, (5, 5), strides=(2, 2), padding='same')(x)
    return Model(x_in, x, name=name)

class SpectralConv2D(Layer):
    '''FNO-style spectral convolution: rfft2 -> truncate to `modes` low frequencies ->
    learned COMPLEX channel-mixing weight -> pad back -> irfft2. Verified locally
    (pure numpy) before writing this: finite output, correct shape, sane scale.
    Complex weight = two real (trainable) tensors combined via tf.complex at call time,
    which keeps gradients well-defined in TF.'''
    def __init__(self, out_channels, modes=12, **kwargs):
        super().__init__(**kwargs)
        self.out_channels = out_channels
        self.modes = modes

    def build(self, input_shape):
        in_ch = input_shape[-1]
        self.w_real = self.add_weight(shape=(self.modes, self.modes, in_ch, self.out_channels),
                                      initializer='glorot_uniform', trainable=True, name='w_real')
        self.w_imag = self.add_weight(shape=(self.modes, self.modes, in_ch, self.out_channels),
                                      initializer='glorot_uniform', trainable=True, name='w_imag')

    def call(self, x):
        H, W = x.shape[1], x.shape[2]
        x_chfirst = tf.transpose(x, [0, 3, 1, 2])                    # (B,C,H,W)
        x_ft = tf.signal.rfft2d(x_chfirst)                           # (B,C,H,W//2+1) complex
        x_ft = tf.transpose(x_ft, [0, 2, 3, 1])                      # (B,H,W//2+1,C)

        m1 = min(self.modes, H); m2 = min(self.modes, x_ft.shape[2])
        x_ft_trunc = x_ft[:, :m1, :m2, :]

        Wc = tf.complex(self.w_real[:m1, :m2], self.w_imag[:m1, :m2])
        out_ft_trunc = tf.einsum('bhwi,hwio->bhwo', x_ft_trunc, Wc)

        pad_h = H - m1; pad_w = (W // 2 + 1) - m2
        out_ft = tf.pad(out_ft_trunc, [[0, 0], [0, pad_h], [0, pad_w], [0, 0]])
        out_ft = tf.transpose(out_ft, [0, 3, 1, 2])
        out = tf.signal.irfft2d(out_ft, fft_length=[H, W])           # (B,C_out,H,W) real
        return tf.transpose(out, [0, 2, 3, 1])                       # (B,H,W,C_out)

print('Shared shell + SpectralConv2D ready')


In [ ]:
# Cell 7 -- Architecture A: SIREN-body (sin() activations, targets spectral bias)
# FIX vs the original Kaggle-notebook version: Keras 3's Functional API rejects a raw
# tf.sin(...) call directly on a KerasTensor ("KerasTensor cannot be used as input to a
# TensorFlow function") -- confirmed by a local CPU run. Wrapped in a Lambda layer instead,
# which is the idiomatic way to apply an arbitrary TF op inside a Functional-API graph.
def siren_conv(x, filters, is_first=False, omega=30.0):
    in_ch = x.shape[-1]
    limit = (1.0 / in_ch) if is_first else (np.sqrt(6.0 / in_ch) / omega)
    init = tf.keras.initializers.RandomUniform(-limit, limit)
    x = Conv2D(filters, 1, padding='same', kernel_initializer=init, bias_initializer=init)(x)
    if is_first:
        return Lambda(lambda t: tf.sin(omega * t), output_shape=lambda s: s)(x)
    return Lambda(lambda t: tf.sin(t), output_shape=lambda s: s)(x)

def siren_block(x, filters):
    skip = x
    x = siren_conv(x, filters)
    x = siren_conv(x, filters)
    return Add()([x, skip])

def siren_body(x, filters, n_blocks):
    x = siren_conv(x, filters, is_first=True, omega=30.0)
    for _ in range(n_blocks):
        x = siren_block(x, filters)
    return x

print('SIREN body ready')


In [ ]:
# Cell 8 -- Architecture B: FNO-body (spectral convolution, global receptive field)
# No fix needed here: every op is already a proper Keras layer (SpectralConv2D, Conv2D,
# Add, BatchNormalization, Activation), none of them raw tf.* calls on a KerasTensor.
def fno_block(x, filters, modes=12):
    skip = x
    spec = SpectralConv2D(filters, modes=modes)(x)      # frequency-domain path
    local = Conv2D(filters, 1, padding='same')(x)         # parallel spatial path (standard FNO design)
    x = Add()([spec, local])
    x = BatchNormalization()(x); x = Activation('relu')(x)
    x = Add()([x, skip])
    return x

def fno_body(x, filters, n_blocks):
    for _ in range(n_blocks):
        x = fno_block(x, filters)
    return x

print('FNO body ready')


In [ ]:
# Cell 9 -- Architecture C: Window-Attention-body (local neighborhood self-attention,
# a message-passing/graph-style local aggregation adapted to a fixed image grid --
# NOT the literature's literal antenna-element graph, see Cell 0's scope note)
# FIX vs the original Kaggle-notebook version: Keras 3 rejects raw tf.reshape/tf.transpose
# on a KerasTensor. The whole windowing + MHA + unwindowing sequence is now a real Layer
# (same pattern as SpectralConv2D) -- raw tf ops are fine inside a Layer's call() because
# they operate on concrete tensors there, not on the outer symbolic Functional-API graph.
class WindowAttentionBlock(Layer):
    def __init__(self, filters, window=8, num_heads=2, **kwargs):
        super().__init__(**kwargs)
        self.filters = filters
        self.window = window
        self.num_heads = num_heads

    def build(self, input_shape):
        C = input_shape[-1]
        self.mha = tf.keras.layers.MultiHeadAttention(num_heads=self.num_heads, key_dim=max(C // self.num_heads, 1))
        self.bn = BatchNormalization()
        self.proj = Conv2D(self.filters, 1, padding='same')
        super().build(input_shape)

    def call(self, x):
        B = tf.shape(x)[0]; H = x.shape[1]; W = x.shape[2]; C = x.shape[-1]
        window = self.window
        skip = x
        xw = tf.reshape(x, [B, H // window, window, W // window, window, C])
        xw = tf.transpose(xw, [0, 1, 3, 2, 4, 5])
        xw = tf.reshape(xw, [-1, window * window, C])
        attn = self.mha(xw, xw)
        attn = tf.reshape(attn, [B, H // window, W // window, window, window, C])
        attn = tf.transpose(attn, [0, 1, 3, 2, 4, 5])
        attn = tf.reshape(attn, [B, H, W, C])
        out = self.bn(attn)
        out = self.proj(out)
        out = Add()([out, skip])
        return Activation('relu')(out)

def window_attention_body(x, filters, n_blocks):
    for _ in range(n_blocks):
        x = WindowAttentionBlock(filters)(x)
    return x

print('Window-attention body ready')


In [ ]:
# Cell 10 -- Architecture D: Gridless-Unfold-body (learned complex soft-threshold refinement)
# Fixes PIA-Net's known bug: soft-threshold the MAGNITUDE, preserve the PHASE exactly,
# instead of ReLU-clamping to real-nonnegative (which cannot represent an arbitrary-phase
# path gain, e.g. a purely-imaginary value).
# FIX vs the original Kaggle-notebook version: a raw tf.Variable created inside a plain
# Python function is NOT reliably tracked as a trainable weight by a Keras 3 Functional
# model (and raw tf.sqrt/tf.stack/tf.reshape on a KerasTensor errors outright, same as
# Cell 7/9). `thresh` is now a proper add_weight on a real Layer -- correctly tracked and
# trained, verified locally (a real gradient step ran end-to-end on CPU before this was
# written into the notebook).
class GridlessUnfoldBlock(Layer):
    def __init__(self, filters, **kwargs):
        super().__init__(**kwargs)
        self.filters = filters

    def build(self, input_shape):
        self.conv = Conv2D(self.filters, 5, padding='same')
        self.bn = BatchNormalization()
        self.thresh = self.add_weight(shape=(), initializer=tf.keras.initializers.Constant(0.1),
                                       trainable=True, name='soft_thresh')
        super().build(input_shape)

    def call(self, x):
        skip = x
        z = self.conv(x)
        z = self.bn(z)
        re = z[..., 0::2]; im = z[..., 1::2]                    # even/odd channels as (real,imag) pairs
        mag = tf.sqrt(re**2 + im**2 + 1e-6)
        new_mag = tf.nn.relu(mag - tf.nn.softplus(self.thresh))  # softplus keeps threshold positive, still learnable
        scale = new_mag / (mag + 1e-6)
        re_out = re * scale; im_out = im * scale
        stacked = tf.stack([re_out, im_out], axis=-1)            # interleave back -- verified locally with numpy
        out = tf.reshape(stacked, tf.shape(z))
        out = out + skip
        return tf.nn.relu(out)

def gridless_unfold_body(x, filters, n_blocks):
    for _ in range(n_blocks):
        x = GridlessUnfoldBlock(filters)(x)
    return x

print('Gridless-unfold body ready')


In [ ]:
# Cell 11 -- Build all 4 models, report parameter counts
ARCHS = {
    'SIREN': siren_body,
    'FNO': fno_body,
    'WindowAttention': window_attention_body,
    'GridlessUnfold': gridless_unfold_body,
}

models = {}
for name, body_fn in ARCHS.items():
    m = build_model_with_body(body_fn, name=name)
    models[name] = m
    print(f'{name:>16}: {m.count_params():>10,} params')
if teacher is not None:
    print(f'{"Teacher (ref)":>16}: {teacher.count_params():>10,} params  (64 blocks, pretrained -- NOT an apples-to-apples depth comparison)')


In [ ]:
# Cell 12 -- Evaluation function (reuses the ORIGINAL evaluator, same convention as every
# other notebook in this project) + the eval set this run will use (full bank by default)
def evaluate_on_bank(model, data_arr, feat_arr, meta_arr, batch_size=8, max_deg_error=1.0):
    detector = get_blob_detector()
    results_by_snr = {}
    N = data_arr.shape[0]
    for start in range(0, N, batch_size):
        end = min(start + batch_size, N)
        preds = model(data_arr[start:end], training=False)
        for j in range(end - start):
            idx = start + j
            L = int(meta_arr[idx, 0]); snr = int(meta_arr[idx, 1])
            peaks, amps = get_blob_peaks(preds[j], detector)
            order = np.argsort(-amps); peaks = peaks[order[:L]]
            angles_est = peaks_to_angles(peaks, sigma=SIGMA, grid_size=M)
            gt_angles, pred_angles = prepare_for_metric(angles_est, feat_arr[idx])
            results_by_snr.setdefault(snr, []).append((gt_angles, pred_angles))
    final_pd, final_rmse = {}, {}
    for snr, examples in results_by_snr.items():
        good_all, bad_all = [], []
        for gt, pred in examples:
            if np.isnan(pred).any(): continue
            diffs = get_ang_difference(gt, pred)
            good, bad = filter_angles(diffs, max_deg_error=max_deg_error)
            good_all.append(good); bad_all.append(bad)
        good_all = np.concatenate(good_all) if good_all else np.array([])
        bad_all = np.concatenate(bad_all) if bad_all else np.array([])
        total = len(good_all) + len(bad_all)
        final_pd[snr] = len(good_all)/total if total > 0 else np.nan
        final_rmse[snr] = np.sqrt(np.mean(good_all**2)) if len(good_all) > 0 else np.nan
    return final_pd, final_rmse

def mean_pd(pd_dict):
    return float(np.nanmean(list(pd_dict.values())))

def subsample_per_snr(data, feat, meta, n_per_snr, block_size=1000, n_blocks=8):
    idx = np.concatenate([np.arange(i*block_size, i*block_size+n_per_snr) for i in range(n_blocks)])
    return data[idx], feat[idx], meta[idx]

if N_PER_SNR_EVAL is None:
    SUB_DATA, SUB_FEAT, SUB_META = EVAL_DATA, EVAL_FEAT, EVAL_META
else:
    SUB_DATA, SUB_FEAT, SUB_META = subsample_per_snr(EVAL_DATA, EVAL_FEAT, EVAL_META, N_PER_SNR_EVAL)
print(f'Evaluation set ready: {SUB_DATA.shape[0]} samples')


In [ ]:
# Cell 12b -- MEASURE real eval throughput (informational -- there's no hard reserve to
# protect in this notebook, unlike the Kaggle version, since there's no overall time cap).
# Still worth knowing up front: evaluate_on_bank's cost is dominated by per-sample CPU
# blob detection + Hungarian matching, not GPU inference, so a small real benchmark on all
# 4 architectures (+ teacher) gives an honest ETA for the full-bank evaluation in Cell 15,
# instead of leaving you to discover it only after training all 4 finishes.
BENCH_N = min(16, SUB_DATA.shape[0])
bench_models = list(models.items())
if teacher is not None:
    bench_models = bench_models + [('Teacher', teacher)]

per_model_sec_per_sample = {}
for name, model in bench_models:
    t0 = time.time()
    evaluate_on_bank(model, SUB_DATA[:BENCH_N], SUB_FEAT[:BENCH_N], SUB_META[:BENCH_N])
    dt = time.time() - t0
    per_model_sec_per_sample[name] = dt / BENCH_N
    print(f'  [{name}] measured: {dt:.1f}s for {BENCH_N} samples -> {dt/BENCH_N*1000:.0f} ms/sample')

projected_minutes = sum(per_model_sec_per_sample.values()) * SUB_DATA.shape[0] / 60
print(f'Projected Cell 15 full-eval time at {SUB_DATA.shape[0]} samples x {len(bench_models)} models: '
      f'~{projected_minutes:.1f} min (measured on this machine, not guessed)')


In [ ]:
# Cell 13 -- Training helpers: disposable batch-size probe, dataset iterator, and the
# per-architecture training loop with measured ETA + periodic checkpointing.
def try_batch(body_fn, batch):
    try:
        probe = build_model_with_body(body_fn, name='probe')
        opt = tf.keras.optimizers.Adam(1e-3)
        x = tf.random.normal((batch, 64, 64, 2)); y = tf.random.normal((batch, M, M, 1))
        with tf.GradientTape() as tape:
            pred = probe(x, training=True)
            loss = tf.reduce_mean(tf.square(pred - y))
        grads = tape.gradient(loss, probe.trainable_variables)
        opt.apply_gradients(zip(grads, probe.trainable_variables))
        del probe, opt
        return True
    except tf.errors.ResourceExhaustedError:
        return False

def make_train_ds(batch):
    def fn():
        for d, g in training_data_generator(sigma=SIGMA, M=M):
            yield d, g
    ds = tf.data.Dataset.from_generator(
        fn, output_signature=(tf.TensorSpec(shape=(64,64,2), dtype=tf.float32),
                              tf.TensorSpec(shape=(M,M,1), dtype=tf.float32)))
    return ds.batch(batch).prefetch(tf.data.AUTOTUNE)

def train_architecture(name, model, body_fn, epochs, steps_per_epoch, ckpt_path,
                        ckpt_every=CHECKPOINT_EVERY_EPOCHS, eta_after_epoch=ETA_AFTER_EPOCH):
    batch = 8
    for candidate in [32, 16, 8]:
        if try_batch(body_fn, candidate):
            batch = candidate; break
    opt = tf.keras.optimizers.Adam(1e-3)   # fresh optimizer for the real model, untouched by the probe

    @tf.function
    def train_step(x, y):
        with tf.GradientTape() as tape:
            pred = model(x, training=True)
            loss = tf.reduce_mean(tf.square(pred - y))
        grads = tape.gradient(loss, model.trainable_variables)
        opt.apply_gradients(zip(grads, model.trainable_variables))
        return loss

    ds_iter = iter(make_train_ds(batch))
    run_t0 = time.time()
    history = []
    for epoch in range(epochs):
        t0 = time.time()
        epoch_losses = []
        for step in range(steps_per_epoch):
            x, y = next(ds_iter)
            epoch_losses.append(float(train_step(x, y)))
        history.append(float(np.mean(epoch_losses)))
        print(f'  [{name}] epoch {epoch+1}/{epochs}: loss={history[-1]:.5f}  '
              f'({time.time()-t0:.1f}s, batch={batch}, total elapsed {elapsed_hours():.2f}h)')
        if epoch + 1 == eta_after_epoch:
            measured_sec_per_epoch = (time.time() - run_t0) / (epoch + 1)
            eta_hours = measured_sec_per_epoch * epochs / 3600
            print(f'  [{name}] MEASURED on this machine: {measured_sec_per_epoch:.1f}s/epoch -> '
                  f'projected {eta_hours:.2f}h for all {epochs} epochs of this architecture')
        if (epoch + 1) % ckpt_every == 0 or (epoch + 1) == epochs:
            model.save_weights(ckpt_path)
    return history, batch

print('Training loop ready (probe uses a throwaway model, real model starts clean)')


In [ ]:
# Cell 14 -- Train + incrementally evaluate all 4 architectures (no wall-clock cap).
# Resilience: if architecture <name>'s ".done.json" marker + weights already exist (e.g.
# from a previous run that got interrupted after this one finished), skip re-training it,
# load its weights, and evaluate it directly instead -- so an interrupted multi-hour/day
# run can be resumed by just re-running the notebook from the top.
histories = {}
batches_used = {}
results = {}
RESULTS_PATH = os.path.join(OUT_DIR, 'architecture_fulleval_results.json')

def save_results_so_far():
    out = {
        'n_blocks': N_BLOCKS, 'epochs_per_arch': EPOCHS_PER_ARCH, 'steps_per_epoch': STEPS_PER_EPOCH,
        'n_eval_samples': int(SUB_DATA.shape[0]), 'total_elapsed_hours': elapsed_hours(),
        'results': {k: {'mean_pd': v['mean_pd'], 'params': int(v['params']),
                        'pd': {str(s): float(p) for s, p in v['pd'].items()},
                        'rmse': {str(s): float(r) for s, r in v['rmse'].items()}}
                   for k, v in results.items()},
        'batches_used': batches_used,
    }
    with open(RESULTS_PATH, 'w') as f:
        json.dump(out, f, indent=2)

for name, body_fn in ARCHS.items():
    ckpt_path = os.path.join(CKPT_DIR, f'{name}.weights.h5')
    done_path = os.path.join(CKPT_DIR, f'{name}.done.json')
    model = models[name]

    if os.path.exists(done_path) and os.path.exists(ckpt_path):
        print(f'=== {name}: found a previous completed run, loading weights (skipping training) ===')
        model.load_weights(ckpt_path)
        with open(done_path) as f:
            done_meta = json.load(f)
        histories[name] = done_meta.get('history', [])
        batches_used[name] = done_meta.get('batch', 8)
    else:
        print(f'\n=== Training {name} ({EPOCHS_PER_ARCH} epochs x {STEPS_PER_EPOCH} steps) ===')
        hist, batch = train_architecture(name, model, body_fn, EPOCHS_PER_ARCH, STEPS_PER_EPOCH, ckpt_path)
        histories[name] = hist; batches_used[name] = batch
        with open(done_path, 'w') as f:
            json.dump({'history': hist, 'batch': batch, 'finished_at_elapsed_h': elapsed_hours()}, f, indent=2)
        print(f'  [{name}] training done, checkpoint + done-marker saved.')

    print(f'  [{name}] evaluating on {SUB_DATA.shape[0]} samples...')
    pd_, rmse_ = evaluate_on_bank(model, SUB_DATA, SUB_FEAT, SUB_META, batch_size=batches_used[name])
    results[name] = {'pd': pd_, 'rmse': rmse_, 'mean_pd': mean_pd(pd_), 'params': model.count_params()}
    print(f'  [{name}] mean Pd = {results[name]["mean_pd"]:.4f}  ({model.count_params():,} params)')
    save_results_so_far()   # incremental save -- survives an interruption before the next architecture

print()
print('All 4 architectures trained + evaluated. Elapsed:', round(elapsed_hours(), 2), 'h')

plt.figure(figsize=(8,4))
for name, hist in histories.items():
    if hist:
        plt.plot(hist, label=name)
plt.xlabel('Epoch'); plt.ylabel('Loss'); plt.title(f'Training loss, all 4 architectures ({EPOCHS_PER_ARCH} epochs each)')
plt.legend(); plt.grid(alpha=.3); plt.tight_layout()
plt.savefig(os.path.join(OUT_DIR, 'fulleval_loss_curves.png'), dpi=130)
plt.show()


In [ ]:
# Cell 15 -- Final comparison: all 4 trained architectures vs teacher, on the full eval bank
if teacher is not None and 'Teacher(ref,64blk)' not in results:
    print('Evaluating teacher (reference row)...')
    t_pd, t_rmse = evaluate_on_bank(teacher, SUB_DATA, SUB_FEAT, SUB_META)
    results['Teacher(ref,64blk)'] = {'pd': t_pd, 'rmse': t_rmse, 'mean_pd': mean_pd(t_pd), 'params': teacher.count_params()}
    save_results_so_far()

print('\n' + '='*60)
print(f'FULL-EVAL VERDICT ({N_BLOCKS}-block, {EPOCHS_PER_ARCH} epochs each, mean Pd across SNR, '
      f'{SUB_DATA.shape[0]} eval samples)')
print('='*60)
ranked = sorted([(k, v) for k, v in results.items() if 'Teacher' not in k], key=lambda kv: -kv[1]['mean_pd'])
for i, (name, r) in enumerate(ranked, 1):
    print(f'{i}. {name:<18} mean Pd = {r["mean_pd"]:.4f}  ({r["params"]:,} params)')
if 'Teacher(ref,64blk)' in results:
    t = results['Teacher(ref,64blk)']
    print(f'\n{"Teacher(ref,64blk)":<18} mean Pd = {t["mean_pd"]:.4f}  ({t["params"]:,} params)'
          f'  <- NOT equal-depth (64 vs {N_BLOCKS} blocks), reference only')
print('\n(This ran the FULL eval bank and a fixed, equal epoch count per architecture --')
print(' a fairer comparison than the Kaggle screening notebook, though still 8 blocks, not 64.)')


In [ ]:
# Cell 16 -- Final save (results were already saved incrementally after each architecture;
# this just re-confirms the final file and prints where everything lives)
save_results_so_far()
print(f'Results: {RESULTS_PATH}')
print(f'Checkpoints: {CKPT_DIR}')
print(f'Loss curves: {os.path.join(OUT_DIR, "fulleval_loss_curves.png")}')
print(f'\nTotal wall-clock time: {elapsed_hours():.2f}h')
